# Гипотеза 1. Высшее образование связано с более высокими зарплатными ожиданиями

Этот ноутбук опирается на уже подготовленный файл `dataset_cleaned.csv`.

Проверяем гипотезу:

**H1:** соискатели с высшим образованием (`education_type = "Высшее"`) ожидают более высокую зарплату, чем соискатели с другими известными уровнями образования.

Критерий подтверждения:
- медианная зарплата в группе с высшим образованием должна быть выше;
- 95% bootstrap CI для разницы медиан должен лежать выше 0;
- перестановочный тест для разницы медиан должен давать малое `p-value`.

In [1]:
import os
from pathlib import Path

os.environ["MPLCONFIGDIR"] = str(Path(".mplconfig"))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

rng = np.random.default_rng(42)

Matplotlib is building the font cache; this may take a moment.


In [2]:
def load_clean_df():
    df = pd.read_csv("dataset_cleaned.csv", parse_dates=["date_publish"], low_memory=False)
    numeric_cols = [
        "salary",
        "experience",
        "birthday",
        "relocation",
        "business_trips",
        "retraining_capability",
        "inner_info_fullness_rate",
        "schedule_type_1",
        "schedule_type_2",
        "schedule_type_3",
        "schedule_type_4",
        "schedule_type_5",
        "schedule_type_6",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "region_group" in df.columns:
        df["region_primary_group"] = (
            df["region_group"]
            .astype("string")
            .str.split("|")
            .str[0]
        )
    if "date_publish" in df.columns and "birthday" in df.columns:
        birthday = pd.to_numeric(df["birthday"], errors="coerce")
        df["age"] = df["date_publish"].dt.year - birthday
        df.loc[~df["age"].between(14, 80), "age"] = np.nan
    return df


def bootstrap_ci_diff(x, y, stat_func=np.median, n_boot=2000, seed=42):
    rng_local = np.random.default_rng(seed)
    x = np.asarray(x)
    y = np.asarray(y)
    boot = np.empty(n_boot)
    for i in range(n_boot):
        xb = rng_local.choice(x, size=len(x), replace=True)
        yb = rng_local.choice(y, size=len(y), replace=True)
        boot[i] = stat_func(xb) - stat_func(yb)
    return np.quantile(boot, [0.025, 0.975])


def permutation_pvalue(x, y, stat_func=np.median, n_perm=2000, seed=42):
    rng_local = np.random.default_rng(seed)
    x = np.asarray(x)
    y = np.asarray(y)
    observed = stat_func(x) - stat_func(y)
    pooled = np.concatenate([x, y])
    stats = np.empty(n_perm)
    x_len = len(x)
    for i in range(n_perm):
        shuffled = rng_local.permutation(pooled)
        stats[i] = stat_func(shuffled[:x_len]) - stat_func(shuffled[x_len:])
    pvalue = (np.abs(stats) >= abs(observed)).mean()
    return observed, pvalue

In [3]:
df = load_clean_df()

edu_df = df[df["education_type"].notna()].copy()
edu_df["higher_education"] = edu_df["education_type"].eq("Высшее")

print("clean_df shape:", df.shape)
print("subset with known education:", edu_df.shape)

edu_df["education_type"].value_counts().to_frame("count")

clean_df shape: (43227, 66)
subset with known education: (9007, 67)


,count
education_type,
Высшее,3995
Среднее профессиональное,3019
Среднее,1592
Незаконченное высшее,401


In [4]:
education_summary = (
    edu_df.groupby("education_type")["salary"]
    .agg(count="size", median="median", mean="mean")
    .sort_values(["median", "count"], ascending=[False, False])
)

comparison = (
    edu_df.groupby("higher_education")["salary"]
    .agg(count="size", median="median", mean="mean")
    .rename(index={False: "other_known_education", True: "higher_education"})
)

display(education_summary)
display(comparison)

,count,median,mean
education_type,,,
Высшее,3995,25000.0,32043.162453
Среднее профессиональное,3019,25000.0,26554.027824
Среднее,1592,25000.0,25674.618090
Незаконченное высшее,401,25000.0,29286.506234


,count,median,mean
higher_education,,,
other_known_education,5012,25000.0,26493.314246
higher_education,3995,25000.0,32043.162453


In [5]:
plt.figure(figsize=(10, 5))
order = (
    edu_df.groupby("education_type")["salary"]
    .median()
    .sort_values(ascending=False)
    .index
)
sns.boxplot(
    data=edu_df,
    x="education_type",
    y="salary",
    order=order,
    showfliers=False,
)
plt.title("Зарплатные ожидания по типу образования")
plt.xlabel("Тип образования")
plt.ylabel("Желаемая зарплата")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

/var/folders/5y/j_6skz8d0y9ggl6xsh5j5zz80000gn/T/ipykernel_23762/3739292768.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
x = edu_df.loc[edu_df["higher_education"], "salary"].dropna().to_numpy()
y = edu_df.loc[~edu_df["higher_education"], "salary"].dropna().to_numpy()

median_diff_ci = bootstrap_ci_diff(x, y, stat_func=np.median, n_boot=2000, seed=42)
mean_diff_ci = bootstrap_ci_diff(x, y, stat_func=np.mean, n_boot=2000, seed=42)
observed_diff, pvalue = permutation_pvalue(x, y, stat_func=np.median, n_perm=2000, seed=42)

result = pd.Series({
    "n_higher": len(x),
    "n_other": len(y),
    "median_higher": float(np.median(x)),
    "median_other": float(np.median(y)),
    "median_diff": float(observed_diff),
    "median_diff_ci_low": float(median_diff_ci[0]),
    "median_diff_ci_high": float(median_diff_ci[1]),
    "mean_higher": float(np.mean(x)),
    "mean_other": float(np.mean(y)),
    "mean_diff_ci_low": float(mean_diff_ci[0]),
    "mean_diff_ci_high": float(mean_diff_ci[1]),
    "permutation_pvalue": float(pvalue),
})

result

n_higher                3995.000000
n_other                 5012.000000
median_higher          25000.000000
median_other           25000.000000
median_diff                0.000000
median_diff_ci_low         0.000000
median_diff_ci_high        0.000000
mean_higher            32043.162453
mean_other             26493.314246
mean_diff_ci_low        4761.721056
mean_diff_ci_high       6294.082494
permutation_pvalue         1.000000
dtype: float64

In [7]:
is_confirmed = (
    (result["median_diff"] > 0)
    and (result["median_diff_ci_low"] > 0)
    and (result["permutation_pvalue"] < 0.05)
)

if is_confirmed:
    print("Гипотеза подтверждена.")
else:
    print("Гипотеза не подтверждена.")
    print("Ключевая причина: медианы совпадают или различаются недостаточно устойчиво.")

Гипотеза не подтверждена.
Ключевая причина: медианы совпадают или различаются недостаточно устойчиво.


## Вывод

Для уже очищенной выборки гипотеза про явное преимущество **высшего образования** по зарплатным ожиданиям не проходит строгую проверку по медиане.

Это не означает, что образование совсем не связано с зарплатой. По среднему значению группа с высшим образованием может выглядеть сильнее, но для типичного соискателя различие неустойчиво. Значит, следующую гипотезу логично искать в признаках, где эффект должен быть более выраженным: мобильность, формат занятости или регион.